[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/maryrobynbernabe/kulintang-6-inference-with-new-videos)

# Steps
1. Record video using python library (opencv)
2. Detect the tone and create video snippet from recording
3. Extract audio from video segment
4. Featurize the audio
5. Pass the audio through the saved model and print predictions
6. Save the predictions with corresponding timestamps in a csv file

- Video
  |
- pause detection
  |
- Based on pauses, segment video
  |
- Create a list of all the video segments
   |
- Loop over each segment and do the following
   |
- Extract Audio
   |
- Featurize audio
   |
- Pass the Audio through the model
   |
- Record the predition with start time of video segment in a .CSV
  |
- Overlay the detection on the final video and save it

# Set required paths

## Set paths

In [ ]:
video_path = "/kaggle/input/datasets/maryrobynbernabe/kulintang-dataset/Kulintang UP MOT (Set 1)/IMG_0037.MOV"
Segment_output_path = "video_segments"
audio_output_path = "audio.wav"
model_path = "/kaggle/input/notebooks/maryrobynbernabe/kulintang-4-classification-model-development/rf_best_model.pkl"

# Segment video based on the silence intervals

## Functions to extract silent intervals

In [ ]:
def extract_silent_timestamps(input_video_path: str, audio_output_path: str, top_db: int = 20 ):
    """
    Extracts non-silent intervals from the audio of a video file.

    This function processes a video to extract its audio track using the `moviepy` library, then analyzes the audio to find intervals of sound using the `librosa` library. Silent sections are determined based on a decibel threshold, and only non-silent timestamps are returned.

    :param input_video_path: str
        The file path to the input video from which audio will be extracted.

    :param audio_output_path: str
        The file path where the extracted audio will be saved temporarily.

    :param top_db: int, optional
        The threshold in decibels for considering sections as non-silent. Defaults to 20. A lower value means more sensitive detection, identifying quieter sounds as non-silent.

    :return: List[Tuple[float, float]]
        A list of tuples, each representing the start and end times (in seconds) of non-silent intervals detected in the audio.

    Notes:
    - This method extracts and temporarily saves the audio from the video file, which is subsequently analyzed. Ensure that there is enough storage space for this temporary file.
    - The function removes any transient silent intervals that fall under the specified top_db threshold, which helps in focusing on meaningful non-silent parts of the audio.
    - Requires `moviepy` and `librosa` to be installed in the Python environment.
    """
    # Step 1: Extract audio from video using moviepy
    clip = VideoFileClip(input_video_path)
    audio = clip.audio
    audio.write_audiofile(audio_output_path)

    # Step 2: Load audio and detect non-silent intervals using librosa
    y, sr = librosa.load(audio_output_path, sr=None)
    non_silent_intervals = librosa.effects.split(y, top_db=top_db)

    # Convert non-silent intervals to seconds
    non_silent_intervals_sec = [(float(start / sr), float(end / sr)) for start, end in non_silent_intervals]

    return non_silent_intervals_sec

## Function to split the video based on silent intervals

In [ ]:
def split_video(input_file: str, output_prefix: str, output_path: str, intervals: list):
    """
    Splits a video into multiple segments based on specified time intervals.

    :param input_file: The path to the input video file.
    :param output_prefix: The prefix for the output files.
    :param intervals: A list of tuples, each containing (start_time, end_time) in seconds.
    :param metainfo: list of metadata items
    :param metadata: metadata dictionary
    """
    with VideoFileClip(input_file) as video:
        for index, (start, end) in enumerate(intervals):
            # Calculate the duration for the current segment
            duration = end - start

            # Subclip the video
            subclip = video.subclip(start, end)

            # Construct the output filename
            output_file = f"{output_prefix}_part{index+1}.mov"

            # Set the output path
            set_output_path = os.path.join(output_path, output_file)

            # Write the subclip to the file
            subclip.write_videofile(set_output_path, codec='libx264')

# Extract audio from the segments, featurize, and get predition

## Required installations

In [ ]:
!pip show moviepy
!sudo apt install ffmpeg

## Functions to extract audio and corresponding features

In [ ]:
from moviepy.editor import VideoFileClip

# Load the video file
def extract_audio(video_path):
  video = VideoFileClip(video_path)

  # Extract and save the audio
  audio = video.audio
  audio.write_audiofile("audio1.wav")

import librosa
import pandas as pd
import numpy as np

#function to extract features from audion files
def extract_features(audio_path):

    # load the audio file
    y,sr = librosa.load(audio_path,mono=True) #load the audio file
    # extract features
    rmse = librosa.feature.rms(y=y)[0] #compute root-mean-square (RMS) value for each frame
    spec_cent = librosa.feature.spectral_centroid(y=y,sr=sr) #calculate spectral_centroid
    spec_bw = librosa.feature.spectral_bandwidth(y=y,sr=sr) #calculate spectral_bandwith
    rolloff = librosa.feature.spectral_rolloff(y=y,sr=sr) #calculate spectral_rolloff
    zcr = librosa.feature.zero_crossing_rate(y) #calculate zero crossing rate
    mfcc = librosa.feature.mfcc(y=y,sr=sr) #Mel frequency ceptral coefficients (mfcc)


    audio_dict={
        'RMSE':rmse.mean(),
        'SPECTRAL_CENTROID':spec_cent.mean(),
        'SPECTRAL_BANDWIDTH':spec_bw.mean(),
        'ROLLOFF':rolloff.mean(),
        'ZERO_CROSSING_RATE':zcr.mean()
    }

    #add the mfcc values
    for index,mfcc_set in enumerate(mfcc):
        feature_name=f"MFCC_FEATURE_{index}"
        audio_dict[feature_name]=np.mean(mfcc_set)

    return audio_dict

## Split the videos into segments

In [ ]:
import os
os.mkdir("video_segments")

In [ ]:
import os
non_silent_intervals = extract_silent_timestamps(video_path, audio_output_path)
video_name = video_path.split("/")[-1].split(".")[0]
split_video(video_path, video_name, Segment_output_path, non_silent_intervals)

## Removing Noises

In [ ]:
import os
import pickle

Confidence_threshold = 0.1

with open(model_path, 'rb') as file:
    model = pickle.load(file)

segments = os.listdir(Segment_output_path)
predictions = []
for segment in segments:
  segment_path = os.path.join(Segment_output_path, segment)
  extract_audio(segment_path)
  audio_dict = extract_features("audio1.wav")
  input = pd.DataFrame([audio_dict])
  prediction = model.predict_proba(input)
  if max(prediction[0]) > Confidence_threshold:
    predictions.append(np.argmax(prediction[0]))
    print(prediction)

In [ ]:
print(predictions)
non_silent_intervals

## Record the predictions with their intervals

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'start_time': [interval[0] for interval in non_silent_intervals],
    'end_time': [interval[1] for interval in non_silent_intervals],
    'prediction': predictions
})
df.to_csv("predictions.csv", index = False)

## Overlay the prediction on the video

In [ ]:
import cv2
import pandas as pd

# Load the CSV with float seconds directly
df = pd.read_csv('predictions.csv')  # Columns: start_time, end_time, prediction

# Open the video file
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise IOError(f"Cannot open video file: {video_path}")

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Define the output video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_video.mp4', fourcc, fps, (width, height))

# Process frames
frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    current_sec = frame_idx / fps

    # Get predictions active at this time
    active_predictions = df[(df['start_time'] <= current_sec) & (df['end_time'] >= current_sec)]

    # Overlay text
    y_offset = 50
    for _, row in active_predictions.iterrows():
        text = f"Prediction: {row['prediction']}"
        cv2.putText(frame, text, (50, y_offset), cv2.FONT_HERSHEY_SIMPLEX,
                    1, (0, 255, 0), 2, cv2.LINE_AA)
        y_offset += 40

    out.write(frame)
    frame_idx += 1

# Clean up
cap.release()
out.release()
print("✅ Video processing complete. Output saved as output_video.mp4")

## Combine output with the audio and save the final video

In [ ]:
!ffmpeg -i output_video.mp4 -i "/kaggle/input/datasets/maryrobynbernabe/kulintang-dataset/Kulintang UP MOT (Set 1)/IMG_0037.MOV" -c:v copy -map 0:v:0 -map 1:a:0 -shortest final_output_with_audio.mp4

In [ ]:
from IPython.display import Video
Video("/kaggle/working/final_output_with_audio.mp4", embed=True)


## Requirements for converting audio into music notation
1. getting the exact notes being played
2. able to identify noise and neglect that
3. accounting for the pauses. we need to understand the rest amount.
4. Converting notes from a music into a pdf document.
5. Able to undestand the hand used for playing (Only for physical playing) - Optional